In [1]:
"""
Compute FID for beta-VAE models using the same method as the DDPM notebook.
Uses pytorch-fid with dims=2048 and 2048 generated samples.
"""

import os
import subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import save_image
from torchvision import transforms
import shutil

try:
    from pytorch_fid import fid_score
except ImportError:
    subprocess.check_call(["pip", "install", "pytorch-fid", "-q"])
    from pytorch_fid import fid_score

# ============================================================
# CONFIG
# ============================================================

CAT_DATASET_PATH = "/kaggle/input/datasets/crawford/cat-dataset"
CATSDOGS_DATASET_PATH = "/kaggle/input/datasets/mikoajrowicki/cats-and-dogs/train"

FID_REAL_CATS = "/kaggle/working/fid_real_cats"
FID_REAL_CATSDOGS = "/kaggle/working/fid_real_catsdogs"

MODELS = {
    "beta_vae_cats_200ep": {
        "weights": "/kaggle/input/datasets/mikoajrowicki/beta-vae/best_model_final_200epochs.pth",
        "latent_dim": 32,
        "fid_real_dir": FID_REAL_CATS,
        "dataset_path": CAT_DATASET_PATH,
    },
    "beta_vae_cats_and_dogs": {
        "weights": "/kaggle/input/datasets/mikoajrowicki/beta-vae/best_model_cats_and_dogs.pth",
        "latent_dim": 32,
        "fid_real_dir": FID_REAL_CATSDOGS,
        "dataset_path": CATSDOGS_DATASET_PATH,
    },
}

N_SAMPLES = 2048
BATCH_SIZE = 64
IMAGE_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# beta-VAE architecture (from notebook cell 3)
# ============================================================

class BetaVAE(nn.Module):
    def __init__(self, latent_dim=64):
        super(BetaVAE, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        self.decoder_input = nn.Linear(latent_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1), nn.Sigmoid()
        )

    def decode(self, z):
        h = self.decoder_input(z)
        h = h.view(-1, 256, 4, 4)
        return self.decoder(h)


# ============================================================
# Helpers
# ============================================================

def ensure_fid_real(real_dir, dataset_path, n_samples, image_size):
    if os.path.exists(real_dir) and len(os.listdir(real_dir)) >= n_samples:
        print(f"  fid_real already exists ({len(os.listdir(real_dir))} imgs), reusing: {real_dir}")
        return
    print(f"  Creating fid_real from {dataset_path} -> {real_dir} ...")
    os.makedirs(real_dir, exist_ok=True)
    from PIL import Image
    import glob

    paths = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        paths.extend(glob.glob(os.path.join(dataset_path, '**', ext), recursive=True))
    paths.sort()

    count = 0
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])
    for p in paths[:n_samples]:
        try:
            img = Image.open(p).convert('RGB')
            save_image(transform(img), os.path.join(real_dir, f"{count:05d}.png"))
            count += 1
        except Exception:
            continue
    print(f"  Saved {count} real images")


def compute_fid_for_vae(model_name, weights_path, latent_dim, real_dir):
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")

    model = BetaVAE(latent_dim=latent_dim).to(DEVICE)
    state_dict = torch.load(weights_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    gen_dir = f"/kaggle/working/fid_generated_{model_name}"
    if os.path.exists(gen_dir):
        shutil.rmtree(gen_dir)
    os.makedirs(gen_dir)

    generated = 0
    with torch.no_grad():
        while generated < N_SAMPLES:
            n = min(BATCH_SIZE, N_SAMPLES - generated)
            z = torch.randn(n, latent_dim).to(DEVICE)
            imgs = model.decode(z)
            for i in range(n):
                save_image(imgs[i], f"{gen_dir}/{generated + i:05d}.png")
            generated += n
    print(f"  Generated {generated} images")

    fid_value = fid_score.calculate_fid_given_paths(
        [real_dir, gen_dir],
        batch_size=50,
        device=DEVICE,
        dims=2048,
    )
    print(f"  FID = {fid_value:.2f}")

    shutil.rmtree(gen_dir)
    return fid_value


# ============================================================
# Main
# ============================================================

results = {}
for name, cfg in MODELS.items():
    if not os.path.exists(cfg["weights"]):
        print(f"\n[SKIP] {name}: weights not found at {cfg['weights']}")
        continue
    ensure_fid_real(cfg["fid_real_dir"], cfg["dataset_path"], N_SAMPLES, IMAGE_SIZE)
    fid = compute_fid_for_vae(name, cfg["weights"], cfg["latent_dim"], cfg["fid_real_dir"])
    results[name] = fid

print(f"\n{'='*60}")
print("  SUMMARY")
print(f"{'='*60}")
for name, fid in results.items():
    print(f"  {name}: FID = {fid:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

  Creating fid_real from /kaggle/input/datasets/crawford/cat-dataset -> /kaggle/working/fid_real_cats ...
  Saved 2048 real images

  beta_vae_cats_200ep
  Generated 2048 images
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 246MB/s]
100%|██████████| 41/41 [00:07<00:00,  5.34it/s]


  FID = 252.50
  Creating fid_real from /kaggle/input/datasets/mikoajrowicki/cats-and-dogs/train -> /kaggle/working/fid_real_catsdogs ...
  Saved 2048 real images

  beta_vae_cats_and_dogs
  Generated 2048 images


100%|██████████| 41/41 [00:08<00:00,  5.12it/s]


  FID = 235.99

  SUMMARY
  beta_vae_cats_200ep: FID = 252.50
  beta_vae_cats_and_dogs: FID = 235.99
